# Scoping Agentic Implementations: Workshop Notebook

This notebook guides you through the scoping process for an agentic implementation:

1. **Persona Mapping** — define who will use the agent
2. **Question Taxonomy** — brainstorm what they will ask
3. **Synthetic Expansion** — use AI_COMPLETE to grow the dataset
4. **Native Eval Dataset** — register with Cortex Agent Evaluations
5. **Multi-Tenancy Assessment** — scope access control requirements
6. **Post-Deploy Observability** — classify real usage into your taxonomy

**Prerequisites:** Run `setup.sql` first to create the lab database and stage.

In [ ]:
# Connection setup
import os

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    from snowflake.snowpark import Session
    connection_params = {
        "account": os.environ.get("SNOWFLAKE_ACCOUNT"),
        "user": os.environ.get("SNOWFLAKE_USER"),
        "password": os.environ.get("SNOWFLAKE_PASSWORD"),
        "role": os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN"),
        "warehouse": "SCOPING_LAB_WH",
        "database": "SCOPING_LAB",
        "schema": "PUBLIC",
    }
    session = Session.builder.configs(connection_params).create()

session.sql("USE DATABASE SCOPING_LAB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
session.sql("USE WAREHOUSE SCOPING_LAB_WH").collect()
print(f"Connected as: {session.sql('SELECT CURRENT_USER()').collect()[0][0]}")
print(f"Role: {session.sql('SELECT CURRENT_ROLE()').collect()[0][0]}")

---
## Activity 1: Persona Mapping

Fill in the persona cards below. Replace the example content with your own use case.
Pick **one persona** as your Phase 1 target.

### Persona 1 (Phase 1 Target)

| Attribute | Your Input |
|-----------|------------|
| **Role** | _e.g., Regional Sales Manager_ |
| **Responsibility** | _e.g., Territory allocation, quota decisions_ |
| **Data Literacy** | _e.g., Reads dashboards, doesn't write SQL_ |
| **Current Workflow** | _e.g., Asks analyst team, 2-day turnaround_ |
| **Frequency** | _e.g., 5-10 questions/week_ |
| **Stakes** | _e.g., Quota misallocation costs $500K/quarter_ |
| **Success Signal** | _e.g., "I got the answer without filing a ticket"_ |

### Persona 2 (Phase 2+)

| Attribute | Your Input |
|-----------|------------|
| **Role** | _e.g., VP of Marketing_ |
| **Responsibility** | _e.g., Budget allocation, board reporting_ |
| **Data Literacy** | _e.g., Needs narrative, not raw numbers_ |
| **Current Workflow** | _e.g., Weekly meeting with analytics lead_ |
| **Frequency** | _e.g., 2-3 questions/week, spikes at quarter-end_ |
| **Stakes** | _e.g., Wrong budget decision = $2M misallocation_ |
| **Success Signal** | _e.g., "I can prep for board meetings without my analyst"_ |

---
## Activity 2: Question Taxonomy

Brainstorm 15-20 questions your Phase 1 persona would actually ask.
Categorize each by type and risk. This becomes the seed for your eval dataset.

In [ ]:
# Define your question taxonomy. Replace these examples with YOUR use case.
from collections import Counter

seed_questions = [
    # LOOKUP (simple fact retrieval from structured data)
    {"question": "What was Q2 revenue for the West region?", "category": "lookup", "risk": "medium"},
    {"question": "How many deals closed last month?", "category": "lookup", "risk": "medium"},
    {"question": "Who is our largest customer by ARR?", "category": "lookup", "risk": "low"},
    
    # AGGREGATION (comparisons, trends across dimensions)
    {"question": "Compare YoY growth by product line", "category": "aggregation", "risk": "high"},
    {"question": "Which region has the highest win rate this quarter?", "category": "aggregation", "risk": "high"},
    {"question": "Show pipeline conversion rates by stage for the last 6 months", "category": "aggregation", "risk": "medium"},
    
    # REASONING (multi-step, requires interpretation or combining sources)
    {"question": "Why did the Northeast region underperform last quarter?", "category": "reasoning", "risk": "high"},
    {"question": "What factors are driving the increase in deal cycle time?", "category": "reasoning", "risk": "high"},
    
    # POLICY (document retrieval — unstructured sources)
    {"question": "What's our discount approval process for enterprise deals?", "category": "policy", "risk": "very_high"},
    {"question": "What are the territory assignment criteria?", "category": "policy", "risk": "high"},
    
    # OUT OF SCOPE (agent should refuse)
    {"question": "Write a cold outreach email for this prospect", "category": "out_of_scope", "risk": "reputational"},
    {"question": "What's our competitor's pricing?", "category": "out_of_scope", "risk": "reputational"},
]

print(f"Seed questions: {len(seed_questions)}")
print(f"\nBreakdown by category:")
for cat, count in Counter(q['category'] for q in seed_questions).items():
    print(f"  {cat}: {count}")

---
## Activity 3: Synthetic Dataset Expansion

Use `SNOWFLAKE.CORTEX.AI_COMPLETE` to expand the seed dataset in two ways:
1. **New questions** — generate net-new questions per category based on the taxonomy
2. **Variations** — rephrase existing questions to test robustness

This quickly takes a 12-question seed to 40-60 questions without manual effort.

In [ ]:
# Generate NEW questions for each category based on the taxonomy
import json

categories_with_examples = {}
for q in seed_questions:
    cat = q['category']
    if cat not in categories_with_examples:
        categories_with_examples[cat] = []
    categories_with_examples[cat].append(q['question'])

generated_questions = []

for category, examples in categories_with_examples.items():
    examples_str = '\n'.join(f'  - {e}' for e in examples)
    
    prompt = f"""You are helping build an evaluation dataset for a sales insights AI agent.

Category: {category}
Existing examples:
{examples_str}

Generate 3 NEW questions a Regional Sales Manager would ask that fit this category.
The questions should be distinct from the examples — cover different metrics, time periods, or dimensions.
For each question, also assign a risk level (low, medium, or high) based on how critical the accuracy of the answer is:
- low: general informational queries, non-critical
- medium: operational decisions, moderate business impact
- high: strategic decisions, financial impact, compliance-related

Return a JSON object with a 'questions' key containing an array of objects, each with 'question' and 'risk' keys."""

    response_format = {
        "type": "json",
        "schema": {
            "type": "object",
            "properties": {
                "questions": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "question": {"type": "string"},
                            "risk": {"type": "string", "enum": ["low", "medium", "high"]}
                        },
                        "required": ["question", "risk"]
                    }
                }
            },
            "required": ["questions"]
        }
    }
    options = json.dumps({"response_format": response_format})

    result = session.sql(f"""
        SELECT AI_COMPLETE('claude-haiku-4-5', $${prompt}$$,
            PARSE_JSON($${options}$$)) AS response
    """).collect()
    
    try:
        response_text = result[0][0]
        if response_text is None:
            print(f"  Warning: NULL response for {category}, skipping")
            continue
        parsed = json.loads(response_text)
        new_qs = parsed["questions"]
        for q in new_qs:
            generated_questions.append({"question": q["question"], "category": category, "risk": q["risk"], "source": "synthetic_new"})
    except (json.JSONDecodeError, KeyError, TypeError) as e:
        print(f"  Warning: Could not parse response for {category}: {e}, skipping")

print(f"Generated {len(generated_questions)} new questions across {len(categories_with_examples)} categories")
for cat, count in Counter(q['category'] for q in generated_questions).items():
    print(f"  {cat}: {count}")

In [ ]:
generated_questions

In [ ]:
# Generate VARIATIONS (rephrasings) of existing questions to test robustness

variation_questions = []

# Pick a subset of seed questions to generate variations for
questions_to_vary = seed_questions[:6]  # first 6 questions

for q in questions_to_vary:
    prompt = f"""Rephrase this question 3 different ways. Each rephrasing should:
- Ask for the same information but with different wording
- Vary formality, specificity, or phrasing style
- Test whether the agent handles natural language variation

Original: {q['question']}

Also assign a risk level (low, medium, or high) to each rephrased question based on how critical the accuracy of the answer is:
- low: general informational queries, non-critical
- medium: operational decisions, moderate business impact
- high: strategic decisions, financial impact, compliance-related

Return a JSON object with a 'variations' key containing an array of objects, each with 'question' and 'risk' keys."""

    response_format = {
        "type": "json",
        "schema": {
            "type": "object",
            "properties": {
                "variations": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "question": {"type": "string"},
                            "risk": {"type": "string", "enum": ["low", "medium", "high"]}
                        },
                        "required": ["question", "risk"]
                    }
                }
            },
            "required": ["variations"]
        }
    }
    options = json.dumps({"response_format": response_format})

    result = session.sql(f"""
        SELECT AI_COMPLETE('claude-haiku-4-5', $${prompt}$$,
            PARSE_JSON($${options}$$)) AS response
    """).collect()
    
    try:
        response_text = result[0][0]
        if response_text is None:
            print(f"  Warning: NULL response for '{q['question'][:40]}...', skipping")
            continue
        parsed = json.loads(response_text)
        variations = parsed["variations"]
        for v in variations:
            variation_questions.append({
                "question": v["question"],
                "category": q['category'],
                "risk": v["risk"],
                "source": "variation",
                "original": q['question']
            })
    except (json.JSONDecodeError, KeyError, TypeError) as e:
        print(f"  Warning: Could not parse response for '{q['question'][:40]}...': {e}, skipping")

print(f"Generated {len(variation_questions)} variations from {len(questions_to_vary)} seed questions")

In [ ]:
# Combine all questions into the full dataset

all_questions = (
    [{**q, "source": "manual"} for q in seed_questions]
    + generated_questions
    + variation_questions
)

print(f"\nFull dataset: {len(all_questions)} questions")
print(f"\nBy source:")
for src, count in Counter(q['source'] for q in all_questions).items():
    print(f"  {src}: {count}")
print(f"\nBy category:")
for cat, count in Counter(q['category'] for q in all_questions).items():
    print(f"  {cat}: {count}")

---
## Activity 4: Native Eval Dataset Creation

Convert the expanded question set into a native Cortex Agent Evaluation dataset.

The native format requires:
- `INPUT_QUERY` (VARCHAR) — the question
- `GROUND_TRUTH` (VARIANT) — JSON object with `ground_truth_output`, `intent`, and `process`

We use `AI_COMPLETE` to draft ground-truth descriptions, then register the dataset.

In [ ]:
# Create the eval questions table in native format
session.sql("""
    CREATE OR REPLACE TABLE EVAL_QUESTIONS (
        INPUT_QUERY   VARCHAR,
        GROUND_TRUTH  VARIANT
    )
""").collect()

print("Created EVAL_QUESTIONS table (INPUT_QUERY VARCHAR, GROUND_TRUTH VARIANT)")

In [ ]:
# Use AI_COMPLETE to draft ground_truth objects for each question
# Then insert into the eval table

# Map categories to process (complexity tier)
category_to_process = {
    "lookup": "single_tool",
    "aggregation": "single_tool",
    "reasoning": "multi_tool",
    "policy": "single_tool",
    "out_of_scope": "refusal",
}

response_format = {
    "type": "json",
    "schema": {
        "type": "object",
        "properties": {
            "ground_truth_output": {"type": "string"},
            "risk": {"type": "string", "enum": ["low", "medium", "high"]}
        },
        "required": ["ground_truth_output", "risk"]
    }
}
options = json.dumps({"response_format": response_format})

inserted = 0
for q in all_questions:
    process = category_to_process.get(q['category'], 'single_tool')
    
    prompt = f"""Generate a ground-truth description for evaluating an AI sales insights agent.

Question: {q['question']}
Category: {q['category']}
Process: {process}

Write a ground-truth description (2-3 sentences) in the 'ground_truth_output' field that:
- Describes what a correct answer SHOULD contain
- Notes what it should NOT contain
- Is specific enough to validate but flexible for non-deterministic responses

Also assign a risk level (low, medium, or high) based on how critical the accuracy of the answer is:
- low: general informational queries, non-critical
- medium: operational decisions, moderate business impact
- high: strategic decisions, financial impact, compliance-related

Return a JSON object with 'ground_truth_output' and 'risk' keys."""

    result = session.sql(f"""
        SELECT AI_COMPLETE('claude-haiku-4-5', $${prompt}$$,
            PARSE_JSON($${options}$$)) AS gt
    """).collect()
    
    try:
        response_text = result[0][0]
        if response_text is None:
            print(f"  Warning: NULL response for '{q['question'][:40]}...', skipping")
            continue
        parsed = json.loads(response_text)
        ground_truth_output = parsed["ground_truth_output"]
        risk = parsed.get("risk", q.get("risk", "medium"))
    except (json.JSONDecodeError, KeyError, TypeError) as e:
        print(f"  Warning: Could not parse response for '{q['question'][:40]}...': {e}, skipping")
        continue
    
    ground_truth_json = json.dumps({
        "ground_truth_output": ground_truth_output,
        "intent": q['category'],
        "process": process,
        "risk": risk
    })
    
    session.sql(f"""
        INSERT INTO EVAL_QUESTIONS
        SELECT $${q['question']}$$, PARSE_JSON($${ground_truth_json}$$)
    """).collect()
    inserted += 1

print(f"Inserted {inserted} questions into EVAL_QUESTIONS")
session.sql("""
    SELECT 
        GROUND_TRUTH:process::VARCHAR AS PROCESS,
        GROUND_TRUTH:intent::VARCHAR AS INTENT,
        COUNT(*) AS N
    FROM EVAL_QUESTIONS
    GROUP BY ALL
    ORDER BY PROCESS, INTENT
""").show()

In [ ]:
SELECT 
    INPUT_QUERY,
    GROUND_TRUTH:intent::VARCHAR AS INTENT,
    GROUND_TRUTH:process::VARCHAR AS PROCESS,
    GROUND_TRUTH:risk::VARCHAR AS RISK,
    GROUND_TRUTH:ground_truth_output::VARCHAR AS GROUND_TRUTH_OUTPUT
FROM EVAL_QUESTIONS
ORDER BY INTENT, RISK DESC

In [ ]:
# Register as a native Cortex Agent evaluation dataset

session.sql("DROP DATASET IF EXISTS SCOPING_LAB.PUBLIC.SCOPING_EVAL_DATASET").collect()

session.sql("""
CALL SYSTEM$CREATE_EVALUATION_DATASET(
  'Cortex Agent',
  'SCOPING_LAB.PUBLIC.EVAL_QUESTIONS',
  'SCOPING_LAB.PUBLIC.SCOPING_EVAL_DATASET',
  OBJECT_CONSTRUCT(
    'query_text', 'INPUT_QUERY',
    'expected_tools', 'GROUND_TRUTH'
  )
)
""").collect()

print("Dataset registered: SCOPING_EVAL_DATASET")
session.sql("SHOW DATASETS IN SCHEMA SCOPING_LAB.PUBLIC").show()

In [ ]:
# Write the evaluation config YAML and upload to stage
# Adjust agent_name to match your agent once built
'''
import tempfile

eval_config_yaml = """
# Cortex Agent Evaluation Configuration
# Scoping Workshop — Baseline

evaluation:
  agent_params:
    agent_name: "YOUR_AGENT_NAME"  # Replace with your Phase 1 agent
    agent_type: "CORTEX AGENT"
  run_params:
    label: "Scoping workshop baseline"
  source_metadata:
    type: "dataset"
    dataset_name: "SCOPING_EVAL_DATASET"

metrics:
  - "answer_correctness"
  - "logical_consistency"
  - name: "tool_selection"
    score_ranges:
      min_score: [1, 3]
      median_score: [4, 6]
      max_score: [7, 10]
    prompt: |
      Evaluate whether the agent selected the correct tool(s) for the user's query.

      User query: {{input}}
      Tools used: {{tool_info}}
      Expected behavior: {{ground_truth}}
      Agent response: {{output}}

      Rate from 1-10:
      1-3 = Wrong tool selected or unnecessary tool calls
      4-6 = Partially correct (right primary tool but missed secondary, or redundant calls)
      7-10 = Optimal tool selection for the query intent

      For out-of-scope questions, score 7-10 only if the agent refused without calling tools.
""".strip()

with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
    f.write(eval_config_yaml)
    yaml_path = f.name

session.sql(f"""
    PUT 'file://{yaml_path}' @SCOPING_LAB.PUBLIC.EVAL_STAGE
    AUTO_COMPRESS=FALSE OVERWRITE=TRUE
""").collect()

print("Eval config uploaded to @EVAL_STAGE")
session.sql("LIST @SCOPING_LAB.PUBLIC.EVAL_STAGE").show()
'''

In [ ]:
# Start an evaluation run (uncomment once you have a deployed agent)
# Replace YOUR_AGENT_NAME in the YAML config first.

# session.sql("""
# CALL EXECUTE_AI_EVALUATION(
#   'START',
#   OBJECT_CONSTRUCT('run_name', 'scoping-baseline-v1'),
#   '@SCOPING_LAB.PUBLIC.EVAL_STAGE/eval_config.yaml'
# )
# """).collect()
# print("Evaluation started. Check status with:")
# print("  CALL EXECUTE_AI_EVALUATION('STATUS', OBJECT_CONSTRUCT('run_name', 'scoping-baseline-v1'))")

print("Eval config ready. Once your Phase 1 agent is deployed:")
print("  1. Update agent_name in the YAML config")
print("  2. Re-upload to stage")
print("  3. Uncomment and run EXECUTE_AI_EVALUATION above")

---
## Activity 5: Multi-Tenancy Scoping

If your agent will serve multiple tenants (different teams, customers, or business units)
with different data access, you need to scope access control into your agent spec.

Answer the questions below to determine your multi-tenancy requirements.

### Multi-Tenancy Decision Matrix

| Question | Your Answer |
|----------|-------------|
| **Will multiple user groups share one agent?** | _Yes / No_ |
| **How many distinct tenants/groups?** | _e.g., 5 teams / 500 customers / 10K external users_ |
| **Do tenants have Snowflake accounts?** | _Yes (internal) / No (external via app)_ |
| **Which tables have tenant-specific data?** | _e.g., SALES_DATA (filter by region), CUSTOMERS (filter by owner)_ |
| **Which columns contain sensitive data?** | _e.g., SALARY, SSN, COMMISSION_RATE_ |
| **How is tenant identity determined?** | _e.g., app JWT claim, Snowflake role, session attribute_ |
| **Does the agent use Cortex Search?** | _Yes / No (RAPs don't apply to search — needs separate approach)_ |

---

### Access Control Strategy

Based on your answers above, fill in:

| Component | Your Spec |
|-----------|----------|
| **Tenant Model** | _single-tenant / multi-tenant-internal / multi-tenant-external_ |
| **Identity Mechanism** | _session attributes (immutable) / Snowflake roles / app-layer filtering_ |
| **Session Attributes** | _e.g., tenant_id, user_role, region_ |
| **Row Access Policies** | _which tables, filter column, policy logic_ |
| **Column Masking** | _which columns, which tenants see full vs masked_ |
| **Entitlements Table** | _Yes / No — for dynamic user-to-tenant mapping_ |
| **Cortex Search Approach** | _per-tenant search service / pre-filtered source / N/A_ |

In [ ]:
# Example: Entitlements table pattern for multi-tenant agents
# This shows how user identity maps to data access without per-user DDL

session.sql("""
    SELECT * FROM ENTITLEMENTS ORDER BY TENANT_ID, USER_ID
""").show()

print("""\nWith this entitlements table + a Row Access Policy:
- The agent runs as a single service account
- Your app passes tenant_id as an immutable session attribute
- The RAP filters every query to only show that tenant's data
- No per-user Snowflake accounts, roles, or grants needed

Example RAP logic:
  CREATE ROW ACCESS POLICY tenant_filter AS (region VARCHAR)
  RETURNS BOOLEAN ->
    region IN (
      SELECT allowed_region FROM ENTITLEMENTS
      WHERE tenant_id = SYS_CONTEXT('CORTEX_AGENT', 'tenant_id')
    );
""")

---
## Post-Deploy: Observability-Driven Iteration

Once your agent is deployed (even to a pilot group), use `CORTEX_AGENT_USAGE_HISTORY`
to see what users actually ask. Since observability data doesn't include category labels,
use `AI_COMPLETE` to classify questions into your taxonomy post-hoc.

In [ ]:
# Query real agent observability data
# Replace AGENT_NAME with your deployed agent

# This shows the actual schema of CORTEX_AGENT_USAGE_HISTORY
print("""Query pattern for real observability data:

SELECT
    METADATA:'input'::STRING AS user_question,
    METADATA:'interaction_interface'::STRING AS surface,
    TOKENS_USED,
    CREDITS_USED,
    START_TIME,
    USER_NAME
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
WHERE AGENT_NAME = 'YOUR_AGENT_NAME'
  AND START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
ORDER BY START_TIME DESC
LIMIT 100;
""")

# Uncomment to run against real data:
# session.sql("""
#     SELECT
#         METADATA:'input'::STRING AS user_question,
#         METADATA:'interaction_interface'::STRING AS surface,
#         TOKENS_USED,
#         START_TIME
#     FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
#     WHERE AGENT_NAME = 'YOUR_AGENT_NAME'
#       AND START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
#     ORDER BY START_TIME DESC
#     LIMIT 100
# """).show()

In [ ]:
# Use AI_COMPLETE to classify observed questions into your taxonomy categories
# This shows the pattern — in production, run this over real observability data

sample_observed_questions = [
    "What were total bookings in EMEA for Q3?",
    "Can you draft a proposal for the Johnson account?",
    "How does our pipeline compare month over month?",
    "What's the refund policy for annual contracts?",
    "Show me the top 10 deals by value this quarter",
]

categories_list = list(categories_with_examples.keys())

response_format = {
    "type": "json",
    "schema": {
        "type": "object",
        "properties": {
            "category": {"type": "string", "enum": categories_list},
            "risk": {"type": "string", "enum": ["low", "medium", "high"]}
        },
        "required": ["category", "risk"]
    }
}
options = json.dumps({"response_format": response_format})

for question in sample_observed_questions:
    prompt = f"""Classify this question into exactly one category and assign a risk level.

Categories: {', '.join(categories_list)}

Question: {question}

Risk levels:
- low: general informational queries, non-critical
- medium: operational decisions, moderate business impact
- high: strategic decisions, financial impact, compliance-related

Return a JSON object with 'category' and 'risk' keys."""
    
    result = session.sql(f"""
        SELECT AI_COMPLETE('claude-haiku-4-5', $${prompt}$$,
            PARSE_JSON($${options}$$)) AS category
    """).collect()
    
    try:
        response_text = result[0][0]
        if response_text is None:
            print(f"  [NULL response ] {question}")
            continue
        parsed = json.loads(response_text)
        classified_cat = parsed["category"]
        risk = parsed["risk"]
        print(f"  [{classified_cat:<13}] [{risk:<6}] {question}")
    except (json.JSONDecodeError, KeyError, TypeError) as e:
        print(f"  [parse error   ] {question}: {e}")

In [ ]:
# At scale: classify all observed questions and compare distribution to your taxonomy
# This SQL pattern runs classification inline over real observability data

print("""At-scale classification pattern (run over real CORTEX_AGENT_USAGE_HISTORY):

WITH observed AS (
    SELECT
        METADATA:'input'::STRING AS user_question,
        TOKENS_USED
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
    WHERE AGENT_NAME = 'YOUR_AGENT_NAME'
      AND START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
),
classified AS (
    SELECT
        user_question,
        TOKENS_USED,
        PARSE_JSON(
            AI_COMPLETE(
                'claude-haiku-4-5',
                'Classify into one category (lookup, aggregation, reasoning, policy, out_of_scope) '
                || 'and assign a risk level (low, medium, high). '
                || 'Return a JSON object with category and risk keys. Question: ' || user_question,
                PARSE_JSON('{"response_format": {"type": "json", "schema": {"type": "object", "properties": {"category": {"type": "string", "enum": ["lookup", "aggregation", "reasoning", "policy", "out_of_scope"]}, "risk": {"type": "string", "enum": ["low", "medium", "high"]}}, "required": ["category", "risk"]}}}')
            )
        ) AS classification
    FROM observed
)
SELECT
    classification:category::VARCHAR AS predicted_category,
    classification:risk::VARCHAR AS risk_level,
    COUNT(*) AS question_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct,
    ROUND(AVG(TOKENS_USED), 0) AS avg_tokens
FROM classified
GROUP BY ALL
ORDER BY question_count DESC;

---
Compare this output to your expected distribution from the taxonomy.
Gaps between expected and actual drive your next iteration:
  - Unexpected categories = new eval questions needed
  - High-token questions = agent confusion, add to eval set
  - Surprise personas = scope Phase 3 multi-tenancy
""")

---
## Agent Spec Document (Final Output)

Fill in the template below to produce your completed scoping document.
This synthesizes all workshop activities into a single deliverable.

### Agent Specification

| Field | Value |
|-------|-------|
| **Agent Name** | _[your_agent_name]_ |
| **Owner** | _[team and individual responsible]_ |
| **Personas Served** | _[from Activity 1]_ |
| **Question Categories** | _[from Activity 2]_ |
| **Out of Scope** | _[explicit refusal boundaries]_ |
| **Business Success Metric** | _[e.g., "80% self-serve without filing a ticket"]_ |
| **Technical Metrics** | _[e.g., answer_correctness >= 0.75, tool_selection >= 7/10]_ |
| **Eval Dataset** | _[SCOPING_EVAL_DATASET — N questions across M categories]_ |
| **Phase 1 Scope** | _[persona + categories that ship first]_ |
| **Phase 1 Exit Criteria** | _[metrics that must be met for Phase 2]_ |

### Access Control (from Activity 5)

| Field | Value |
|-------|-------|
| **Tenant Model** | _[single / multi-internal / multi-external]_ |
| **Identity Mechanism** | _[session attributes / roles / app-layer]_ |
| **Session Attributes** | _[list attributes passed at API call time]_ |
| **Row Access Policies** | _[tables + filter logic]_ |
| **Column Masking** | _[columns + visibility rules]_ |
| **Cortex Search Tenancy** | _[per-tenant service / pre-filtered / N/A]_ |

---
## Phased Delivery Plan

| Phase | Scope | Entry Criteria | Exit Criteria |
|-------|-------|----------------|---------------|
| **1: Prove Value** | _[1 persona, lookup+aggregation]_ | Prerequisites met | answer_correctness >= 0.75, 5+ pilot users |
| **2: Expand Coverage** | _[add policy/reasoning categories]_ | Phase 1 exit met | tool_selection >= 7/10, correctness >= 0.80 |
| **3: Multi-Persona + Tenancy** | _[add personas, RAP/masking]_ | Phase 2 exit, tenancy spec done | per-tenant evals pass |
| **4: Hardening** | _[budgets, CI/CD gates, SLAs]_ | Phase 3 exit, observability baseline | SLA met 30 consecutive days |

---
## Next Steps

1. **Formalize:** Transfer the spec into your project tracking system
2. **Address blockers:** Build prerequisites (semantic view, search service, entitlements table)
3. **Build Phase 1 agent:** Use the spec as the requirements doc
4. **Run baseline eval:** Execute against your registered dataset
5. **Deploy to pilot:** 5-10 users, observe with CORTEX_AGENT_USAGE_HISTORY
6. **Classify + iterate:** Use AI_COMPLETE to categorize real questions, add gaps to eval set

**Companion modules:**
- `evaluations/` — Full eval lifecycle (GPA framework, CI/CD gates, versioning)
- `cortex-ai-observability/` — Surface identification, cost attribution, budgets
- `agent_versioning/` — Safe iteration with named versions and aliases
- `cortex-agent-multi-tenancy/` — RAP, masking, entitlements implementation
- `agent-routing/` — Multi-agent orchestration for Phase 3+

In [ ]:
# Optional: Clean up (uncomment to run)
# session.sql("DROP DATABASE IF EXISTS SCOPING_LAB").collect()
# session.sql("DROP WAREHOUSE IF EXISTS SCOPING_LAB_WH").collect()